In [3]:
# Install required package
!pip install vecstack

In [4]:
# Import libraries

import pandas as pd
import numpy as np

from vecstack import stacking

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")

In [5]:
# Define file paths

train_file = "data/train.csv"
test_file = "data/test.csv"
output_file = "data/output_regression.xlsx"

In [6]:
# Load datasets

trainData = pd.read_csv(train_file)
testData = pd.read_csv(test_file)

print("Training shape:", trainData.shape)
print("Test shape:", testData.shape)

Training shape: (750, 1001)
Test shape: (150, 1001)


In [7]:
# Separate features and target

y_train = trainData["TARGET"]
X_train = trainData.drop(["TARGET"], axis=1)

y_test = testData["TARGET"]
X_test = testData.drop(["TARGET"], axis=1)

print(X_train.shape)
print(X_test.shape)

(750, 1000)
(150, 1000)


In [8]:
# Decision Tree Regressor

dt_model = DecisionTreeRegressor()

dt_model.fit(X_train, y_train)

dt_train_pred = dt_model.predict(X_train)
dt_test_pred = dt_model.predict(X_test)

print("Decision Tree Train MSE:", mean_squared_error(y_train, dt_train_pred))
print("Decision Tree Test MSE:", mean_squared_error(y_test, dt_test_pred))

#Create prediction dataframe

df_dt = pd.DataFrame()
df_dt["TARGET"] = dt_test_pred

Decision Tree Train MSE: 1.7935222607106133e-36
Decision Tree Test MSE: 0.0024942


In [9]:
# Random Forest Regressor

rf_model = RandomForestRegressor()

rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

print("Random Forest Train MSE:", mean_squared_error(y_train, rf_train_pred))
print("Random Forest Test MSE:", mean_squared_error(y_test, rf_test_pred))

# Create prediction dataframe

df_rf = pd.DataFrame()
df_rf["TARGET"] = rf_test_pred

Random Forest Train MSE: 0.00014645317413333332
Random Forest Test MSE: 0.001080413945333333


In [10]:
# Gradient Boosting Regressor

gb_model = GradientBoostingRegressor()

gb_model.fit(X_train, y_train)

gb_train_pred = gb_model.predict(X_train)
gb_test_pred = gb_model.predict(X_test)

print("Gradient Boosting Train MSE:", mean_squared_error(y_train, gb_train_pred))
print("Gradient Boosting Test MSE:", mean_squared_error(y_test, gb_test_pred))

# Create prediction dataframe

df_gb = pd.DataFrame()
df_gb["TARGET"] = gb_test_pred

Gradient Boosting Train MSE: 7.26640373862041e-05
Gradient Boosting Test MSE: 0.0009910416645816135


In [11]:
# Stacking Ensemble

models = [
    GradientBoostingRegressor(),
    RandomForestRegressor(),
    DecisionTreeRegressor()
]

S_train, S_test = stacking(
    models,
    X_train,
    y_train,
    X_test,
    regression=True,
    mode="oof_pred_bag",
    needs_proba=False,
    save_dir=None,
    n_folds=4,
    verbose=2
)

task:         [regression]
metric:       [mean_absolute_error]
mode:         [oof_pred_bag]
n_models:     [3]

model  0:     [GradientBoostingRegressor]
    fold  0:  [0.02558433]
    fold  1:  [0.01812110]
    fold  2:  [0.01592392]
    fold  3:  [0.01756162]
    ----
    MEAN:     [0.01929774] + [0.00371828]
    FULL:     [0.01930456]

model  1:     [RandomForestRegressor]
    fold  0:  [0.02446170]
    fold  1:  [0.01824410]
    fold  2:  [0.01677786]
    fold  3:  [0.01808428]
    ----
    MEAN:     [0.01939198] + [0.00298175]
    FULL:     [0.01939721]

model  2:     [DecisionTreeRegressor]
    fold  0:  [0.03473936]
    fold  1:  [0.02464362]
    fold  2:  [0.02250802]
    fold  3:  [0.02182353]
    ----
    MEAN:     [0.02592863] + [0.00519213]
    FULL:     [0.02593867]



In [12]:
# Train the stacked model

stack_model = GradientBoostingRegressor()

stack_model.fit(S_train, y_train)

stack_train_pred = stack_model.predict(S_train)
stack_test_pred = stack_model.predict(S_test)

print("Stacking Train MSE:", mean_squared_error(y_train, stack_train_pred))
print("Stacking Test MSE:", mean_squared_error(y_test, stack_test_pred))

# Create prediction dataframe

df_stack = pd.DataFrame()
df_stack["TARGET"] = stack_test_pred

Stacking Train MSE: 0.0004387278336708805
Stacking Test MSE: 0.0008288846596933393


In [13]:
# Save predictions to Excel

# Save predictions from all models to one Excel file

with pd.ExcelWriter(output_file) as writer:
    df_dt.to_excel(writer, sheet_name="DecisionTree", index=False)
    df_rf.to_excel(writer, sheet_name="RandomForest", index=False)
    df_gb.to_excel(writer, sheet_name="GradientBoosting", index=False)
    df_stack.to_excel(writer, sheet_name="Stacking", index=False)

print("Output saved at:", output_file)

Output saved at: data/output_regression.xlsx
